In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from model.model_cifar10 import CIFAR10Classifier
import torch.nn.functional as F

In [2]:
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),  # Randomly flip the image horizontally
    transforms.RandomCrop(32, padding=4),  # Randomly crop
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))  # Normalize using CIFAR-10 stats
])

batch_size = 64
train_dataset = datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.CIFAR10(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

Files already downloaded and verified
Files already downloaded and verified


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CIFAR10Classifier().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [4]:
# 4. Training Loop with Learning Rate Scheduler
def train_model(model, train_loader, criterion, optimizer, device, num_epochs=10):
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)  # Decrease LR by a factor of 0.1 every 5 epochs
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            # Calculate accuracy
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        scheduler.step()
        
        epoch_loss = running_loss / len(train_loader)
        epoch_accuracy = 100 * correct / total
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.2f}%, LR: {scheduler.get_last_lr()[0]:.6f}")


In [5]:
train_model(model, train_loader, criterion, optimizer, device, num_epochs=30)

Epoch [1/30], Loss: 1.6377, Accuracy: 39.95%, LR: 0.001000
Epoch [2/30], Loss: 1.3328, Accuracy: 51.49%, LR: 0.001000
Epoch [3/30], Loss: 1.2070, Accuracy: 57.01%, LR: 0.001000
Epoch [4/30], Loss: 1.1350, Accuracy: 59.55%, LR: 0.001000
Epoch [5/30], Loss: 1.0864, Accuracy: 61.73%, LR: 0.001000
Epoch [6/30], Loss: 1.0463, Accuracy: 63.27%, LR: 0.001000
Epoch [7/30], Loss: 1.0209, Accuracy: 64.36%, LR: 0.001000
Epoch [8/30], Loss: 1.0052, Accuracy: 64.49%, LR: 0.001000
Epoch [9/30], Loss: 0.9784, Accuracy: 65.97%, LR: 0.001000
Epoch [10/30], Loss: 0.9580, Accuracy: 66.32%, LR: 0.000100
Epoch [11/30], Loss: 0.8854, Accuracy: 68.78%, LR: 0.000100
Epoch [12/30], Loss: 0.8746, Accuracy: 69.47%, LR: 0.000100
Epoch [13/30], Loss: 0.8624, Accuracy: 69.79%, LR: 0.000100
Epoch [14/30], Loss: 0.8550, Accuracy: 69.99%, LR: 0.000100
Epoch [15/30], Loss: 0.8512, Accuracy: 70.20%, LR: 0.000100
Epoch [16/30], Loss: 0.8497, Accuracy: 70.30%, LR: 0.000100
Epoch [17/30], Loss: 0.8397, Accuracy: 70.50%, LR

KeyboardInterrupt: 

In [19]:
# 5. Evaluation
def evaluate_model(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    print(f"Test Accuracy: {100 * correct / total:.2f}%")

# Evaluate the model
evaluate_model(model, test_loader, device)

# 6. Save the model
torch.save(model.state_dict(), "model/cifar10_model.pth")

Test Accuracy: 75.97%
